<a href="https://colab.research.google.com/github/satyam-1605/RAG-from-basic-to-advance/blob/main/rag_with_huggingface_and_mongodb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets pandas pymongo sentence_transformers

In [ ]:
!pip install -U transformers accelerate

In [ ]:
from datasets import load_dataset

In [ ]:
import pandas as pd

In [ ]:

#dataset=load_dataset("AIatMongoDB/embedded_movies")
dataset=load_dataset("MongoDB/embedded_movies")
#MongoDB/embedded_movies

In [ ]:
data=pd.DataFrame(dataset["train"])

In [ ]:
data

In [ ]:
data.columns

In [ ]:
data.isnull().sum()

In [ ]:
data["fullplot"][0]

In [ ]:
data.isnull().sum().sum()

In [ ]:
data=data.dropna(subset=["fullplot"])

In [ ]:
data

In [ ]:
data=data.drop(columns=["plot_embedding"])

In [ ]:
data

In [ ]:

from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("thenlper/gte-large")

In [ ]:


text="   sunny savita is  a data scientist who create prodcut of data"


In [ ]:
text.strip()

In [ ]:

def get_embedding(text:str)->list[float]:

  if not text.strip():
    print("attempted to get embedding for empty text.")
    return []

  embedding=embedding_model.encode(text)
  return embedding.tolist()


In [ ]:

data["embedding"]=data["fullplot"].apply(get_embedding)

In [ ]:
import pymongo

In [ ]:
!python --version

In [ ]:

from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

uri = "mongodb_uri"

# Create a new client and connect to the server
client = MongoClient(uri, server_api=ServerApi('1'))

# Send a ping to confirm a successful connection
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

In [ ]:
db=client["mongorag"]

In [ ]:
collection=db["mongoragcollection"]

In [ ]:
collection.insert_one({
    "name":"satyam",
    "age":25,
    "city":"delhi"
})

In [ ]:
data

In [ ]:
document=data.to_dict("records")

In [ ]:
collection.insert_many(document)

In [ ]:

{
 "fields": [{
     "numDimensions": 1024,
     "path": "embedding",
     "similarity": "cosine",
     "type": "vector"
   }]
}

In [ ]:
def get_embedding(text:str)->list[float]:

  if not text.strip():
    print("attempted to get embedding for empty text.")
    return []

  embedding=embedding_model.encode(text)
  return embedding.tolist()


In [ ]:
def vector_search(user_query,collection):

  query_embedding=get_embedding(user_query)
  print(query_embedding)

  if query_embedding is None:
    return "Invalid query or embeddig is failed"

  pipeline=[

            {
                "$vectorSearch":{

                "index": "vector_index",
                "queryVector": query_embedding,
                "path": "embedding",
                "numCandidates": 150,  # Number of candidate matches to consider
                "limit": 4,  # Return top 4 matches


                }

            },

              {
                 "$project":{

                "fullplot": 1,  # Include the plot field
                "title": 1,  # Include the title field
                "genres": 1,  # Include the genres field
                "score": {"$meta": "vectorSearchScore"},  # Include the search score
                 }

            }

           ]

  result=collection.aggregate(pipeline)
  return list(result)

In [ ]:
vector_search("what is the best horror movie to watch and why?",collection)

In [ ]:
def get_search_result(query,collection):

  get_knowledge=vector_search(query,collection)

  search_result=""

  for result in get_knowledge:
        search_result += f"Title: {result.get('title', 'N/A')}, Plot: {result.get('fullplot', 'N/A')}\n"

  return search_result


In [ ]:
query="what is the best comedy movie to watch and why?"

In [ ]:
collection

In [ ]:
source_information=get_search_result(query,collection)


In [ ]:


combined_information = f"Query: {query}\nContinue to answer the query by using the Search Results:\n{source_information}."

print(combined_information)


In [ ]:

from huggingface_hub import notebook_login
notebook_login()


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it")



In [ ]:
model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it", device_map="auto")

In [ ]:
input_ids = tokenizer(combined_information, return_tensors="pt").to("cuda")

In [ ]:
response = model.generate(**input_ids, max_new_tokens=500)

In [ ]:

print(tokenizer.decode(response[0]))